# MLOps - Model Evaluation

In [ ]:
import json
import shutil
from datetime import datetime
from pathlib import Path

import dagshub
import mlflow

import os
from dagshub.auth import add_app_token
from kaggle_secrets import UserSecretsClient

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing import (
    image_dataset_from_directory,
)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# BASE_DIR explicite — fonctionne en local ET sur Kaggle/Colab
BASE_DIR = Path(os.environ.get("PROJECT_BASE_DIR", Path.cwd()))

UPLOADS_DIR = BASE_DIR / "uploads"
DATASETS_DIR = BASE_DIR / "uploads" / "datasets"
MODELS_DIR = BASE_DIR / "models"

PRODUCTION_DIR = (
    MODELS_DIR / "production"
)

EVALUATION_DIR = (
    MODELS_DIR / "evaluations"
)

PRODUCTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

USER_NAME = "ML_Goat"
REPO_NAME = "Citrus_Disease_MLOps"

# Authentification non-interactive (obligatoire en headless)
def get_dagshub_token():
    # Fonctionne si le notebook est exécuté interactivement dans l'éditeur Kaggle
    try:
        return UserSecretsClient().get_secret("DAGSHUB_TOKEN")
    except Exception:
        pass

    # Fallback pour une exécution déclenchée via l'API (kaggle kernels push)
    token_file = Path("/kaggle/input/citrus-secrets/dagshub_token.txt")
    if token_file.exists():
        return token_file.read_text().strip()

    raise RuntimeError("Impossible de récupérer le token DagsHub (ni Secrets, ni dataset privé).")

dagshub_token = get_dagshub_token()
add_app_token(token=dagshub_token)

# Connecter mlflow avec DagsHub
dagshub.init(repo_owner=USER_NAME, repo_name=REPO_NAME, mlflow=True)

MLFLOW_EXPERIMENT_NAME = "Citrus Disease - DenseNet121 Evaluation"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [ ]:
# ============================================================
# PARAMÈTRES
# ============================================================

IMAGE_SIZE = (224, 224)

BATCH_SIZE = 32

SEED = 42

In [ ]:
# ============================================================
# CLASSES
# ============================================================

CLASS_NAMES = [
    "black_spot",
    "canker",
    "greening",
    "healthy",
    "rot",
    "scab",
]

In [ ]:
# ============================================================
# SPLIT
# ============================================================

TRAIN_SIZE = 0.70

VALIDATION_SIZE = 0.15

TEST_SIZE = 0.15

In [ ]:
# ============================================================
# TROUVER LE DERNIER DATASET
# ============================================================

def get_latest_dataset():

    if not DATASETS_DIR.exists():

        raise FileNotFoundError(
            f"Le dossier {DATASETS_DIR} "
            f"n'existe pas."
        )

    versions = []

    for directory in DATASETS_DIR.iterdir():

        if not directory.is_dir():
            continue

        if not directory.name.startswith("v"):
            continue

        try:

            version_number = int(
                directory.name[1:]
            )

            versions.append(
                (version_number, directory)
            )

        except ValueError:
            continue

    if not versions:

        raise FileNotFoundError(
            "Aucun dataset trouvé."
        )

    versions.sort(
        key=lambda x: x[0]
    )

    version_number, directory = versions[-1]

    return (
        f"v{version_number}",
        directory,
    )

In [ ]:
# ============================================================
# TROUVER LE DERNIER MODÈLE
# ============================================================

def get_latest_model():

    if not MODELS_DIR.exists():

        raise FileNotFoundError(
            "Le dossier models n'existe pas."
        )

    models = []

    for file in MODELS_DIR.glob(
        "densenet121_v*.keras"
    ):

        name = file.stem

        try:

            version = int(
                name.split("_v")[-1]
            )

            models.append(
                (version, file)
            )

        except ValueError:
            continue

    if not models:

        raise FileNotFoundError(
            "Aucun modèle trouvé."
        )

    models.sort(
        key=lambda x: x[0]
    )

    version, model_path = models[-1]

    return (
        f"v{version}",
        model_path,
    )

In [ ]:
# ============================================================
# CRÉER LE TEST SET
# ============================================================

def load_test_dataset(
    dataset_dir: Path,
):

    print(
        "\nCréation du test set..."
    )

    # --------------------------------------------------------
    # Même première séparation que train.py
    #
    # 70% train
    # 30% temporary
    # --------------------------------------------------------

    temporary_dataset = (
        image_dataset_from_directory(
            dataset_dir,
            labels="inferred",
            label_mode="int",
            class_names=CLASS_NAMES,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            validation_split=0.30,
            subset="validation",
            seed=SEED,
            shuffle=True,
        )
    )

    # --------------------------------------------------------
    # Le temporary contient 30%.
    #
    # On prend la deuxième moitié :
    #
    # 15% test
    # --------------------------------------------------------

    temporary_batches = (
        tf.data.experimental.cardinality(
            temporary_dataset
        ).numpy()
    )

    if temporary_batches <= 1:

        raise ValueError(
            "Le dataset temporaire est trop petit "
            "pour créer le test set."
        )

    validation_batches = (
        temporary_batches // 2
    )

    test_dataset = (
        temporary_dataset.skip(
            validation_batches
        )
    )

    # --------------------------------------------------------
    # Pas de shuffle pour l'évaluation
    # --------------------------------------------------------

    test_dataset = test_dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return test_dataset

In [ ]:
# ============================================================
# PRÉDICTIONS
# ============================================================

def get_predictions(
    model,
    test_dataset,
):

    y_true = []

    y_pred = []

    for images, labels in test_dataset:

        predictions = model.predict(
            images,
            verbose=0,
        )

        predicted_classes = np.argmax(
            predictions,
            axis=1,
        )

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            predicted_classes
        )

    return (
        np.array(y_true),
        np.array(y_pred),
    )

In [ ]:
# ============================================================
# CALCUL DES MÉTRIQUES
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }

In [ ]:
# ============================================================
# ÉVALUER UN MODÈLE
# ============================================================

def evaluate_model(
    model,
    test_dataset,
):

    y_true, y_pred = get_predictions(
        model,
        test_dataset,
    )

    metrics = calculate_metrics(
        y_true,
        y_pred,
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
    )

    return (
        metrics,
        report,
        matrix,
    )

In [ ]:
# ============================================================
# MODÈLE DE PRODUCTION
# ============================================================

def get_production_model():

    production_model = (
        PRODUCTION_DIR
        / "densenet121_production.keras"
    )

    if not production_model.exists():

        return None

    return production_model

In [ ]:
# ============================================================
# SAUVEGARDER LES RÉSULTATS
# ============================================================

def save_evaluation(
    model_version,
    dataset_version,
    metrics,
    report,
    matrix,
):

    result = {

        "model_version":
            model_version,

        "dataset_version":
            dataset_version,

        "evaluation_set":
            "test_15_percent",

        "metrics":
            metrics,

        "classification_report":
            report,

        "confusion_matrix":
            matrix.tolist(),

        "evaluated_at":
            datetime.now().isoformat(),
    }

    output_file = (
        EVALUATION_DIR
        / f"evaluation_{model_version}.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            result,
            file,
            indent=4,
        )

    print()
    print(
        f"Résultats sauvegardés : "
        f"{output_file}"
    )

    return output_file

In [ ]:
# ============================================================
# PROMOUVOIR UN MODÈLE
# ============================================================

def promote_model(
    candidate_model: Path,
):

    production_model = (
        PRODUCTION_DIR
        / "densenet121_production.keras"
    )

    shutil.copy2(
        candidate_model,
        production_model,
    )

    print()
    print(
        "✅ Nouveau modèle promu "
        "en production."
    )

    print(
        f"Production : "
        f"{production_model}"
    )

In [ ]:
# ============================================================
# AFFICHER LES MÉTRIQUES
# ============================================================

def print_metrics(
    title,
    metrics,
):

    print()
    print("=" * 60)
    print(title)
    print("=" * 60)

    print(
        f"Accuracy  : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Precision : "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Recall    : "
        f"{metrics['recall']:.4f}"
    )

    print(
        f"F1-score  : "
        f"{metrics['f1_score']:.4f}"
    )

In [ ]:
# ============================================================
# MAIN
# ============================================================

def main():

    print()
    print("=" * 60)
    print("       MLOps - Model Evaluation")
    print("=" * 60)

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    dataset_version, dataset_dir = (
        get_latest_dataset()
    )

    print(
        f"\nDataset : "
        f"{dataset_version}"
    )

    print(
        f"Chemin : "
        f"{dataset_dir}"
    )

    # --------------------------------------------------------
    # Test set
    # --------------------------------------------------------

    test_dataset = load_test_dataset(
        dataset_dir
    )

    # --------------------------------------------------------
    # Modèle candidat
    # --------------------------------------------------------

    model_version, model_path = (
        get_latest_model()
    )

    print(
        f"\nModèle candidat : "
        f"{model_version}"
    )

    print(
        f"Chemin : "
        f"{model_path}"
    )

    # --------------------------------------------------------
    # Charger le modèle candidat
    # --------------------------------------------------------

    print(
        "\nChargement du modèle..."
    )

    candidate_model = (
        keras.models.load_model(
            model_path
        )
    )

    # --------------------------------------------------------
    # MLFLOW RUN
    # --------------------------------------------------------

    with mlflow.start_run(run_name=f"evaluate_{model_version}") as run:

        print()
        print(f"MLflow Run ID : {run.info.run_id}")

        mlflow.log_params(
            {
                "model_version": model_version,
                "dataset_version": dataset_version,
                "evaluation_set": "test_15_percent",
                "test_split": TEST_SIZE,
                "num_classes": len(CLASS_NAMES),
            }
        )

        mlflow.set_tags(
            {
                "stage": "evaluation",
                "model_type": "DenseNet121",
                "dataset_version": dataset_version,
                "candidate_model_version": model_version,
            }
        )

        # --------------------------------------------------------
        # Évaluation candidat
        # --------------------------------------------------------

        (
            candidate_metrics,
            candidate_report,
            candidate_matrix,
        ) = evaluate_model(
            candidate_model,
            test_dataset,
        )

        print_metrics(
            "Évaluation du nouveau modèle "
            "sur le TEST SET",
            candidate_metrics,
        )

        print()
        print(
            "Classification Report :"
        )

        print(
            candidate_report
        )

        print(
            "Confusion Matrix :"
        )

        print(
            candidate_matrix
        )

        # --------------------------------------------------------
        # Sauvegarder les résultats
        # --------------------------------------------------------

        evaluation_file = save_evaluation(
            model_version,
            dataset_version,
            candidate_metrics,
            candidate_report,
            candidate_matrix,
        )

        mlflow.log_metrics(
            {
                "candidate_accuracy": candidate_metrics["accuracy"],
                "candidate_precision": candidate_metrics["precision"],
                "candidate_recall": candidate_metrics["recall"],
                "candidate_f1_score": candidate_metrics["f1_score"],
            }
        )

        mlflow.log_artifact(
            str(evaluation_file),
            artifact_path="evaluation",
        )

        # --------------------------------------------------------
        # Chercher le modèle production
        # --------------------------------------------------------

        production_model_path = (
            get_production_model()
        )

        # --------------------------------------------------------
        # PREMIER MODÈLE
        # --------------------------------------------------------

        if production_model_path is None:

            print()
            print(
                "⚠️ Aucun modèle en production."
            )

            print(
                "Le nouveau modèle devient "
                "le modèle de production."
            )

            promote_model(
                model_path
            )
            mlflow.set_tag(
                "promotion_decision",
                "promoted",
            )
            mlflow.set_tag(
                "promotion_decision",
                "promoted_first_model",
            )

            print()
            print(
                "Évaluation terminée."
            )

            return

        # --------------------------------------------------------
        # Charger le modèle production
        # --------------------------------------------------------

        print()
        print(
            "Chargement du modèle actuel..."
        )

        production_model = (
            keras.models.load_model(
                production_model_path
            )
        )

        # --------------------------------------------------------
        # Évaluer le modèle production
        # sur LE MÊME TEST SET
        # --------------------------------------------------------

        (
            production_metrics,
            production_report,
            production_matrix,
        ) = evaluate_model(
            production_model,
            test_dataset,
        )

        print_metrics(
            "Évaluation du modèle actuel "
            "sur le TEST SET",
            production_metrics,
        )

        # --------------------------------------------------------
        # Comparaison
        # --------------------------------------------------------

        candidate_f1 = (
            candidate_metrics["f1_score"]
        )

        production_f1 = (
            production_metrics["f1_score"]
        )

        candidate_accuracy = (
            candidate_metrics["accuracy"]
        )

        production_accuracy = (
            production_metrics["accuracy"]
        )

        mlflow.log_metrics(
            {
                "production_accuracy": production_metrics["accuracy"],
                "production_precision": production_metrics["precision"],
                "production_recall": production_metrics["recall"],
                "production_f1_score": production_metrics["f1_score"],
                "f1_improvement": candidate_f1 - production_f1,
                "accuracy_improvement": candidate_accuracy - production_accuracy,
            }
        )

        print()
        print("=" * 60)
        print("Comparaison des modèles")
        print("=" * 60)

        print(
            f"                    "
            f"Production     Nouveau"
        )

        print(
            f"Accuracy             "
            f"{production_accuracy:.4f}"
            f"          "
            f"{candidate_accuracy:.4f}"
        )

        print(
            f"F1-score             "
            f"{production_f1:.4f}"
            f"          "
            f"{candidate_f1:.4f}"
        )

        # --------------------------------------------------------
        # CRITÈRE DE PROMOTION
        #
        # Pour commencer :
        # le nouveau modèle doit avoir un meilleur F1-score.
        # --------------------------------------------------------

        if candidate_f1 > production_f1:

            print()
            print(
                "✅ Le nouveau modèle "
                "a un meilleur F1-score."
            )

            print(
                "Promotion en production..."
            )

            promote_model(
                model_path
            )

        else:

            print()
            print(
                "❌ Le nouveau modèle "
                "n'est pas meilleur."
            )

            print(
                "Le modèle actuel reste "
                "en production."
            )

        # --------------------------------------------------------
        # FIN
        # --------------------------------------------------------

        print()
        print("=" * 60)
        print("Évaluation terminée.")
        print("=" * 60)
        print()

In [ ]:
# ============================================================
# EXECUTION
# ============================================================

if __name__ == "__main__":
    main()